#### **1. Download SRTM DEM**
#### **2. Processing SRTM DEM**


In [7]:
import geopandas as gpd
from utils.get_dem import get_dem


In [ ]:
path_ygp = 'data/boundary/ygp_region.gpkg'
ygp_vec_gdf = gpd.read_file(path_ygp)
ygp_vec_gdf.head()  
ygp_vec_gdf.bounds  

,minx,miny,maxx,maxy
0,97.217573,20.641807,109.236523,30.668281


In [9]:
lonmin, lonmax, latmin, latmax = 97, 110, 20, 31
for lon in range(lonmin, lonmax, 5):    
    for lat in range(latmin, latmax, 5):        
        dem_out = 'data/dem/tiles/SRTMGL3_{}_{}.tif'.format(lon, lat)
        region = [lon-0.1, lon+5+0.1, lat-0.1, lat+5+0.1]
        print('Ouput dem:', dem_out, 'Region:',  region)
        get_dem(demtype='SRTMGL3', bounds=region, path_out=dem_out)


Ouput dem: data/dem/tiles/SRTMGL3_97_20.tif Region: [96.9, 102.1, 19.9, 25.1]
!!Output file has been existed.
Ouput dem: data/dem/tiles/SRTMGL3_97_25.tif Region: [96.9, 102.1, 24.9, 30.1]
!!Output file has been existed.
Ouput dem: data/dem/tiles/SRTMGL3_97_30.tif Region: [96.9, 102.1, 29.9, 35.1]
!!Output file has been existed.
Ouput dem: data/dem/tiles/SRTMGL3_102_20.tif Region: [101.9, 107.1, 19.9, 25.1]
!!Output file has been existed.
Ouput dem: data/dem/tiles/SRTMGL3_102_25.tif Region: [101.9, 107.1, 24.9, 30.1]
!!Output file has been existed.
Ouput dem: data/dem/tiles/SRTMGL3_102_30.tif Region: [101.9, 107.1, 29.9, 35.1]
!!Output file has been existed.
Ouput dem: data/dem/tiles/SRTMGL3_107_20.tif Region: [106.9, 112.1, 19.9, 25.1]
!!Output file has been existed.
Ouput dem: data/dem/tiles/SRTMGL3_107_25.tif Region: [106.9, 112.1, 24.9, 30.1]
Ouput dem: data/dem/tiles/SRTMGL3_107_30.tif Region: [106.9, 112.1, 29.9, 35.1]


### srtm dem processing  
including dem mosaic, downsampling and clipping.


In [10]:
import os
import rasterio as rio
import geopandas as gpd
from glob import glob
import matplotlib.pyplot as plt
from rasterio.merge import merge
from rasterio.mask import mask
from rasterio.warp import reproject, Resampling

### Mosaic

In [11]:
paths_dem_ls = glob('data/dem/tiles/*')
path_mosaic = 'data/dem/SRTMGL3_90m.tif'
src_files_to_mosaic = []
for fp in paths_dem_ls:
    src = rio.open(fp)
    src_files_to_mosaic.append(src)
mosaic_arr, mosaic_trans = merge(src_files_to_mosaic)
mosaic_meta = src.meta.copy()
mosaic_meta.update({
    "height": mosaic_arr.shape[1],
    "width": mosaic_arr.shape[2],
    "transform": mosaic_trans
    })

# Write the mosaic raster to disk
with rio.open(path_mosaic, 'w', **mosaic_meta) as dest:
    dest.write(mosaic_arr)


### Downsampling

In [12]:
path_srtm = 'data/dem/SRTMGL3_90m.tif'
path_srtm_down = 'data/dem/SRTMGL3_300m.tif' 

with rio.open(path_srtm) as src:
    scale_factor = 300 / 90  
    dst_width, dst_height = int(src.width / scale_factor), int(src.height / scale_factor)    
    dst_transform = src.transform * src.transform.scale(scale_factor, scale_factor)
    dst_meta = src.meta.copy()
    dst_meta.update({
        "width": dst_width,
        "height": dst_height,
        "transform": dst_transform})
    with rio.open(path_srtm_down, "w", **dst_meta) as dst:
        reproject(
            source=rio.band(src, 1),
            destination=rio.band(dst, 1),
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dst_transform,
            dst_crs=src.crs,
            resampling=Resampling.average,  
        )



#### clip to ygp subregions. 

In [13]:
path_dem = 'data/dem/SRTMGL3_300m.tif'
path_dem_save = 'data/dem/SRTMGL3_300m_ygp.tif'
path_ynp = 'data/boundary/ygp_region.gpkg'
hma_vec_gdf = gpd.read_file(path_ynp)

with rio.open(path_dem) as src:
    if hma_vec_gdf.crs != src.crs: 
        hma_vec_gdf = hma_vec_gdf.to_crs(src.crs)      
    for idx, row in hma_vec_gdf.iterrows():
        geom = row.geometry
        clipped_arr, clipped_transform = mask(dataset=src, shapes = [geom], 
                                              crop=True, all_touched=True)
        meta = src.meta.copy()
        meta.update({
            "height": clipped_arr.shape[1],
            "width": clipped_arr.shape[2],
            "transform": clipped_transform})
        ## save to path
        if os.path.exists(path_dem_save): os.remove(path_dem_save)
        with rio.open(path_dem_save, "w", **meta) as dst:
            print(dst.bounds)
            dst.write(clipped_arr)
        print(f"saved to: {path_dem_save}")



BoundingBox(left=97.21624999993703, bottom=20.639305555564498, right=109.23847222215653, top=30.669861111117775)
saved to: data/dem/SRTMGL3_300m_ygp.tif
